# 01 Data Check Clean

This notebook records the analysis implemented in 01 Data Check Clean.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

# DATA3001 - Global Drifter Program data overview

This notebook connects to the NOAA Global Drifter Program hourly dataset, documents its variables, and screens candidate study regions.

**Important:** The global dataset is opened lazily. Do not run `ds.load()` on the complete dataset.

## 1. Configure the secure cloud connection

In [1]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

## 2. Import analysis tools

In [2]:
from clouddrift.datasets import gdp1h
from clouddrift.ragged import subset
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. Open the hourly drifter dataset

`gdp1h()` initially reads only the structure and metadata from cloud storage.

In [3]:
ds = gdp1h()
ds

<xarray.Dataset> Size: 16GB
Dimensions:                (traj: 19396, obs: 197214787)
Coordinates:
    id                     (traj) int64 155kB ...
    time                   (obs) datetime64[ns] 2GB ...
Dimensions without coordinates: traj, obs
Data variables: (12/58)
    BuoyTypeManufacturer   (traj) |S20 388kB ...
    BuoyTypeSensorArray    (traj) |S20 388kB ...
    CurrentProgram         (traj) float64 155kB ...
    DeployingCountry       (traj) |S20 388kB ...
    DeployingShip          (traj) |S20 388kB ...
    DeploymentComments     (traj) int16 39kB ...
    ...                     ...
    start_lat              (traj) float32 78kB ...
    start_lon              (traj) float32 78kB ...
    typebuoy               (traj) |S10 194kB ...
    typedeath              (traj) int8 19kB ...
    ve                     (obs) float32 789MB ...
    vn                     (obs) float32 789MB ...
Attributes: (12/18)
    Conventions:          CF-1.6
    acknowledgement:      Elipot, Shane; Sykulski, Adam; Lumpkin, Rick; Centu...
    contributor_name:     NOAA Global Drifter Program
    contributor_role:     Data Acquisition Center
    date_created:         2024-07-30T19:02:16.200531
    doi:                  10.25921/x46c-3620
    ...                   ...
    publisher_name:       GDP Drifter DAC
    publisher_url:        https://www.aoml.noaa.gov/phod/gdp
    summary:              Global Drifter Program hourly data
    time_coverage_end:    2022-10-31:00:00:00Z
    time_coverage_start:  1987-10-02:13:00:00Z
    title:                Global Drifter Program hourly drifting buoy collection

## 4. Dataset overview

In [4]:
dataset_overview = pd.Series({
    "title": ds.attrs.get("title", ""),
    "summary": ds.attrs.get("summary", ""),
    "processing_level": ds.attrs.get("processing_level", ""),
    "doi": ds.attrs.get("doi", ""),
    "number_of_trajectories": ds.sizes.get("traj", 0),
    "number_of_observations": ds.sizes.get("obs", 0),
})

display(dataset_overview.to_frame("value"))

,value
title,Global Drifter Program hourly drifting buoy co...
summary,Global Drifter Program hourly data
processing_level,Level 2 QC by GDP drifter DAC
doi,10.25921/x46c-3620
number_of_trajectories,19396
number_of_observations,197214787


## 5. Complete variable dictionary

Variables on `obs` describe hourly observations. Variables on `traj` describe complete drifter trajectories.

In [5]:
variable_rows = []

for name, variable in ds.variables.items():
    if "obs" in variable.dims:
        level = "hourly observation"
    elif "traj" in variable.dims:
        level = "trajectory metadata"
    else:
        level = "other"

    variable_rows.append({
        "variable": name,
        "level": level,
        "dimensions": ", ".join(variable.dims),
        "data_type": str(variable.dtype),
        "units": variable.attrs.get("units", ""),
        "description": variable.attrs.get("long_name", ""),
        "standard_name": variable.attrs.get("standard_name", ""),
    })

variable_dictionary = pd.DataFrame(variable_rows)
pd.set_option("display.max_rows", None)
display(variable_dictionary)

,variable,level,dimensions,data_type,units,description,standard_name
0,BuoyTypeManufacturer,trajectory metadata,traj,|S20,-,Buoy type manufacturer,
1,BuoyTypeSensorArray,trajectory metadata,traj,|S20,-,Buoy type sensor array,
2,CurrentProgram,trajectory metadata,traj,float64,-,Current Program,
3,DeployingCountry,trajectory metadata,traj,|S20,-,Deploying country,
4,DeployingShip,trajectory metadata,traj,|S20,-,Name of deployment ship,
5,DeploymentComments,trajectory metadata,traj,int16,-,Deployment comments,
6,DeploymentStatus,trajectory metadata,traj,|S20,-,Deployment status,
7,DragAreaAboveDrogue,trajectory metadata,traj,float32,m^2,Drag area above drogue.,
8,DragAreaOfDrogue,trajectory metadata,traj,float32,m^2,Drag area drogue.,
9,DragAreaRatio,trajectory metadata,traj,float32,m,Drag area ratio,


## 6. Core analysis variables

Current speed can be derived as $\sqrt{v_e^2 + v_n^2}$. Sea-surface temperature is in Kelvin and converts to degrees Celsius by subtracting 273.15.

In [6]:
core_variables = ["id", "time", "lon", "lat", "ve", "vn", "sst", "rowsize"]

core_overview = variable_dictionary[
    variable_dictionary["variable"].isin(core_variables)
].reset_index(drop=True)

display(core_overview)

,variable,level,dimensions,data_type,units,description,standard_name
0,lat,hourly observation,obs,float64,degrees_north,Latitude,
1,lon,hourly observation,obs,float64,degrees_east,Longitude,
2,rowsize,trajectory metadata,traj,int32,-,Number of observations per trajectory,
3,sst,hourly observation,obs,float32,Kelvin,Fitted sea water temperature,
4,ve,hourly observation,obs,float32,m/s,Eastward velocity,
5,vn,hourly observation,obs,float32,m/s,Northward velocity,
6,id,trajectory metadata,traj,int64,-,Global Drifter Program Buoy ID,
7,time,hourly observation,obs,datetime64[ns],,Time,


## 7. Preliminary candidate-region screening

These counts describe drifters whose starting locations fall inside each box. They do not count every observation or every drifter that later passed through the region. The East Australian Current is excluded because it is the course example.

In [7]:
candidates = {
    "Gulf Stream": (-80, -50, 25, 45),
    "Kuroshio": (120, 155, 20, 40),
    "Agulhas": (10, 40, -45, -25),
    "California Current": (-135, -115, 25, 50),
    "Benguela": (5, 20, -35, -15),
    "Brazil-Malvinas": (-60, -40, -50, -30),
    "Bay of Bengal": (80, 100, 5, 25),
}

candidate_rows = []

for region_name, (lon_min, lon_max, lat_min, lat_max) in candidates.items():
    mask = (
        (ds["start_lon"] >= lon_min)
        & (ds["start_lon"] <= lon_max)
        & (ds["start_lat"] >= lat_min)
        & (ds["start_lat"] <= lat_max)
    )

    candidate_rows.append({
        "region": region_name,
        "lon_min": lon_min,
        "lon_max": lon_max,
        "lat_min": lat_min,
        "lat_max": lat_max,
        "starting_drifters": int(mask.sum().compute()),
    })

candidate_summary = (
    pd.DataFrame(candidate_rows)
    .sort_values("starting_drifters", ascending=False)
    .reset_index(drop=True)
)

display(candidate_summary)

,region,lon_min,lon_max,lat_min,lat_max,starting_drifters
0,Gulf Stream,-80,-50,25,45,873
1,Kuroshio,120,155,20,40,721
2,California Current,-135,-115,25,50,365
3,Agulhas,10,40,-45,-25,298
4,Bay of Bengal,80,100,5,25,280
5,Brazil-Malvinas,-60,-40,-50,-30,201
6,Benguela,5,20,-35,-15,173


## Next steps

1. Select two or three candidate regions.
2. Compare observation density, distinct drifters, and time coverage.
3. Choose a defensible region and research question.
4. Load only the selected regional subset.

## 8. Worked example: California Current

Research question: What are the mean surface circulation and
sea-surface temperature patterns in region R during 2020?

In [ ]:
from urllib.parse import quote

variables = [
    "ID",
    "time",
    "longitude",
    "latitude",
    "ve",
    "vn",
    "sst",
    "err_ve",
    "err_vn",
    "err_sst",
    "flg_sst",
    "drogue_lost_date",
]

constraints = [
    "time>=2020-01-01T00:00:00Z",
    "time<=2020-01-31T23:59:59Z",
    "longitude>=-130",
    "longitude<=-118",
    "latitude>=30",
    "latitude<=45",
]

base_url = (
    "https://erddap.aoml.noaa.gov/gdp/erddap/"
    "tabledap/drifter_hourly_qc.csv"
)

query = ",".join(variables) + "&" + "&".join(constraints)
url = base_url + "?" + quote(query, safe=",:&=-")

region_df = pd.read_csv(
    url,
    skiprows=[1]
)

region_df["time"] = pd.to_datetime(region_df["time"], utc=True)

print("\u884c\u6570：", len(region_df))
print("\u4e0d\u540c\u6d6e\u6807\u6570\u91cf：", region_df["ID"].nunique())

display(region_df.head(10))

In [ ]:
analysis_columns = [
    "longitude",
    "latitude",
    "ve",
    "vn",
    "sst",
    "err_ve",
    "err_vn",
    "err_sst",
    "flg_sst",
]

quality_summary = pd.DataFrame({
    "data_type": region_df[analysis_columns].dtypes.astype(str),
    "missing_count": region_df[analysis_columns].isna().sum(),
    "missing_percent": (
        region_df[analysis_columns].isna().mean() * 100
    ).round(2),
})

display(quality_summary)

print("SST\u8d28\u91cf\u6807\u8bb0：")
display(
    region_df["flg_sst"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# Implementation note translated in the organised English notebook.
work_df = region_df.copy()

# Implementation note translated in the organised English notebook.
work_df["speed"] = np.sqrt(
    work_df["ve"]**2 + work_df["vn"]**2
)

# Implementation note translated in the organised English notebook.
work_df["sst_celsius"] = work_df["sst"] - 273.15

# Implementation note translated in the organised English notebook.
current_df = (
    work_df
    .dropna(subset=["longitude", "latitude", "ve", "vn"])
    .copy()
)

# Implementation note translated in the organised English notebook.
temperature_df = (
    work_df[
        work_df["sst"].notna()
        & (work_df["flg_sst"] == 5)
    ]
    .copy()
)

print("\u6d0b\u6d41\u5206\u6790\u8bb0\u5f55\u6570：", len(current_df))
print("\u6e29\u5ea6\u5206\u6790\u8bb0\u5f55\u6570：", len(temperature_df))

display(
    current_df[
        ["ve", "vn", "speed"]
    ].describe()
)

display(
    temperature_df[
        ["sst_celsius"]
    ].describe()
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    constrained_layout=True
)

# Implementation note translated in the organised English notebook.
axes[0].scatter(
    current_df["longitude"],
    current_df["latitude"],
    s=5,
    alpha=0.25,
    color="royalblue"
)

axes[0].set_title("Hourly drifter observations")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_xlim(R["lon"])
axes[0].set_ylim(R["lat"])
axes[0].grid(alpha=0.3)

# Implementation note translated in the organised English notebook.
density = axes[1].hist2d(
    current_df["longitude"],
    current_df["latitude"],
    bins=[24, 30],
    range=[
        [R["lon"][0], R["lon"][1]],
        [R["lat"][0], R["lat"][1]]
    ],
    cmap="viridis",
    cmin=1
)

axes[1].set_title("Observation density")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_xlim(R["lon"])
axes[1].set_ylim(R["lat"])

fig.colorbar(
    density[3],
    ax=axes[1],
    label="Number of hourly observations"
)

plt.show()

In [ ]:
grid_size = 0.5

grid_df = current_df.copy()

grid_df["lon_cell"] = (
    np.floor(grid_df["longitude"] / grid_size) * grid_size
)

grid_df["lat_cell"] = (
    np.floor(grid_df["latitude"] / grid_size) * grid_size
)

cell_summary = (
    grid_df
    .groupby(["lon_cell", "lat_cell"])
    .agg(
        observations=("ID", "size"),
        distinct_drifters=("ID", "nunique"),
        mean_ve=("ve", "mean"),
        mean_vn=("vn", "mean"),
        mean_speed=("speed", "mean"),
    )
    .reset_index()
)

display(
    cell_summary
    .sort_values(
        ["distinct_drifters", "observations"],
        ascending=False
    )
    .head(20)
)